### The ReAct Pattern

* ReAct = Reasoning + Acting. It's a way of making the LLM think out loud in a structured text format — every step is written as a Thought, an Action, an Action Input, and an Observation — so you can read exactly how the agent reasoned its way to the answer.

* Unlike tool-calling agents (which output silent JSON), ReAct agents show their work, making them auditable and debuggable. This pattern is used in production wherever the reasoning trail matters — banking, healthcare, legal, and any AI system that must explain its decisions.

In [1]:
import os
import logging
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import AgentExecutor, create_react_agent

In [2]:
os.environ['ANONYMIZED_TELEMETRY'] = 'False'
logging.getLogger('httpx').setLevel(logging.WARNING)

In [3]:
load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

Two Simple Tools

In [4]:
@tool
def search_crop_disease(query: str) -> str:
    '''
    Search the crop disease knowledge base for disease information and treatment.
    '''
    
    knowledge = {
        'cassava mosaic': 'Cassava Mosaic Disease: viral, causes yellowing and mottling of leaves. Control: use disease-free cuttings, plant resistant varieties, remove infected plants.',
        'maize smut':     'Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, rotate crops, remove galls before they burst.',
        'rice blast':     'Rice Blast: fungal, causes diamond-shaped lesions on leaves. Control: use resistant varieties, balance nitrogen, apply fungicides when needed.',
    }
    
    q = query.lower()
    for key, value in knowledge.items():
        if key in q:
            return value
    return f'No knowledge found for: {query}'
    
@tool
def get_weather(city: str) -> str:
    '''
    Return the current weather for a city.
    '''
    
    fake = {
        'lagos': 'Sunny, 32°C, humidity 78%',
        'abuja': 'Cloudy, 28°C, humidity 65%',
        'kano':  'Hot and dry, 36°C, humidity 30%',
    }
    
    return fake.get(city.lower(), f'No weather data for {city}')

print('Two tools ready')
print(f'   - {search_crop_disease.name}')
print(f'   - {get_weather.name}')

Two tools ready
   - search_crop_disease
   - get_weather


The ReAct Prompt Template

In [5]:
react_template = '''
Answer the following question as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought: {agent_scratchpad}
'''

prompt = PromptTemplate.from_template(react_template)

print('ReAct prompt template ready')


ReAct prompt template ready


Build the ReAct Agent and Executor

In [6]:
tools = [search_crop_disease, get_weather]

# Builds the agent
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,     # stop after 5 loops
    max_execution_time=30,  # stop after 30 seconds
)

print('ReAct agent ready')

ReAct agent ready


ReAct Test

In [7]:
agent_executor.invoke({'input': 'How do I treat maize smut?'})



> Entering new AgentExecutor chain...
I need to find information about maize smut, including its treatment options. I'll search the crop disease knowledge base for this information.

Action: search_crop_disease  
Action Input: "maize smut treatment"  Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, rotate crops, remove galls before they burst.I have found information about maize smut and its treatment options. The disease is fungal and causes grey or black galls on ears and stalks. The recommended control measures include planting resistant hybrids, rotating crops, and removing galls before they burst.

Final Answer: To treat maize smut, plant resistant hybrids, rotate crops, and remove galls before they burst.

> Finished chain.


{'input': 'How do I treat maize smut?',
 'output': 'To treat maize smut, plant resistant hybrids, rotate crops, and remove galls before they burst.'}

In [8]:
agent_executor.invoke({'input': 'How do I treat maize smut in Yoruba?'})



> Entering new AgentExecutor chain...
I need to find information on treating maize smut, specifically in the Yoruba language. I'll use the search_crop_disease tool to look for treatment options for maize smut.

Action: search_crop_disease  
Action Input: "maize smut treatment in Yoruba"  Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, rotate crops, remove galls before they burst.I have found information on maize smut treatment. It involves planting resistant hybrids, rotating crops, and removing galls before they burst. However, I need to translate this information into Yoruba for better understanding.

Action: search_crop_disease  
Action Input: "maize smut treatment in Yoruba"  Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, rotate crops, remove galls before they burst.The information I found does not include a translation into Yoruba. I need to summarize the treatment options 

{'input': 'How do I treat maize smut in Yoruba?',
 'output': '1. Gba irugbin ti o ni idena si aisan.\n2. Yi awọn irugbin pada.\n3. Yọ awọn galls kuro ṣaaju ki wọn to fọ.'}

In [9]:
agent_executor.invoke({'input': 'How do I treat maize smut in both Yoruba, Hausa, Igbo, Pidgin, French and Arbic?'})



> Entering new AgentExecutor chain...
I need to search for information on maize smut treatment in the crop disease knowledge base. 

Action: search_crop_disease  
Action Input: "maize smut treatment"  Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, rotate crops, remove galls before they burst.I have found information on how to treat maize smut. Now, I need to translate this information into Yoruba, Hausa, Igbo, Pidgin, French, and Arabic.

**Translation of Treatment Information:**

1. **Yoruba**: 
   - Maize Smut: arun fungal, n fa galls grẹy tabi dudu lori awọn eti ati awọn stalks. Iṣakoso: gbin awọn hybrids ti o ni resistance, yipada awọn irugbin, yọ galls ṣaaju ki wọn to fọ.

2. **Hausa**: 
   - Maize Smut: cuta ta fungus, tana haifar da galls masu launin toka ko baki a kan ganyen da tsire-tsire. Kulawa: shuka nau'ikan da suka yi juri, canza gonaki, cire galls kafin su fashe.

3. **Igbo**: 
   - Maize Smut: ọrịa fungal, na-eme k

{'input': 'How do I treat maize smut in both Yoruba, Hausa, Igbo, Pidgin, French and Arbic?',
 'output': "- **Yoruba**: Maize Smut: arun fungal, n fa galls grẹy tabi dudu lori awọn eti ati awọn stalks. Iṣakoso: gbin awọn hybrids ti o ni resistance, yipada awọn irugbin, yọ galls ṣaaju ki wọn to fọ.\n- **Hausa**: Maize Smut: cuta ta fungus, tana haifar da galls masu launin toka ko baki a kan ganyen da tsire-tsire. Kulawa: shuka nau'ikan da suka yi juri, canza gonaki, cire galls kafin su fashe.\n- **Igbo**: Maize Smut: ọrịa fungal, na-eme ka galls ojii ma ọ bụ ọcha na ears na stalks. Nlekọta: tinye hybrids nwere mgbochi, gbanwee mkpụrụ, wepụ galls tupu ha agbaji.\n- **Pidgin**: Maize Smut: na fungal disease wey dey cause grey or black galls for ears and stalks. Control: plant resistant hybrids, change crops, remove galls before dem burst.\n- **French**: Maize Smut: maladie fongique, cause des galls gris ou noirs sur les épis et les tiges. Contrôle : planter des hybrides résistants, faire 